# Validation and generalization of the emotion vectors: Parts A, B and C

DSC 180A · Member 2 (Flavia) · reproduces Sofroniew et al. (2026) §1.2 on Qwen3-8B, using Member 1's vectors from extraction run `ab1333f49024705c`.

**The question behind all three parts:** the team's emotion vectors were built from Qwen-written stories. Do they really track *emotion*, beyond the stories used to build them?
Each part moves further away from those stories:

| Part | Asks | Input | Main measurement |
|---|---|---|---|
| **A** | Do the vectors recognize emotion in stories about **topics they never saw**? | 168 validation + 168 test stories (saved activations; no model needed) | Top-1 accuracy and AUROC against a label-shuffle null |
| **B** | Do they respond to a **chat message that implies an emotion** without naming it? | 35 hand-written scenarios + 20 neutral prompts | Rank of the intended emotion (top-1, top-3, mean rank) |
| **C** | When **only a number changes** (e.g. a Tylenol dose), do the scores move with how dire it is? | 6 templates × 10 values, including a "grapes" control | Spearman ρ per line, compared with the grapes control |

All three only **read** the model. They test whether the vectors *detect* emotion, not whether they *cause* behavior; that is the steering experiment's question.

### How to run
| Where | What runs | How |
|---|---|---|
| **Google Colab, GPU runtime (T4)** | A, B and C, all real | Run the setup cell below, set `RUN_DIR`, then *Run all* |
| Laptop without an NVIDIA GPU | A for real; **B and C are skipped** (they print "skipped") | *Run all* |
| Laptop, code check | A for real; B and C with **Qwen3-0.6B and random vectors: CODE CHECK ONLY, not results** | Start Jupyter with `BC_MODE=dryrun` |

Parts B and C need Qwen3-8B in 4-bit NF4, the same setup that produced the vectors, and that needs an NVIDIA CUDA GPU.

**Colab setup** (run once, in a new cell):
```python
from google.colab import drive; drive.mount('/content/drive')
!git clone -b flavia/validation-generalization https://github.com/praveendesharma/dsc180a-steering-llms.git
%cd dsc180a-steering-llms/validation_generalization
```
Then set `RUN_DIR` in §0.2 to the Drive folder holding the unzipped archive.

**Outputs:**
- Part A: `outputs/A/`.
- Parts B and C: `outputs/BC/` (real) or `outputs/BC_DRYRUN_code_check_only/` (code check).
- All of `outputs/` is git-ignored.

## 0. Setup

### 0.1 Install (Colab only)
**What:** on Colab, installs the same pinned library versions notebook 03 used (transformers 4.x, bitsandbytes for 4-bit weights).
**Why:** the vectors were extracted with this exact stack. Reading the model with a different quantization or library version could put its internal states in a slightly different place, and the vectors would no longer fit them. Locally nothing is installed.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install "transformers>=4.51.0,<5" "accelerate>=1.6,<2" "bitsandbytes>=0.45,<1" safetensors scipy

### 0.2 Configuration
**What:** collects every path and setting in one place, and decides whether Parts B/C run (`BC_MODE`).
**Why:**
- **One place for paths.** `RUN_DIR` is the only path most people need to change.
- **Settings fixed in advance.** The block (23), seed (180) and number of permutations (500) are written down before any result is seen. That keeps choices from being tuned to the results.
- **No surprise crashes.** `BC_MODE="auto"` runs B/C only where they can run properly (a CUDA GPU). Otherwise it skips them instead of failing halfway.

In [ ]:
import csv, gc, hashlib, json, re, time, zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import sklearn
from sklearn.metrics import confusion_matrix, roc_auc_score
from scipy.stats import spearmanr
try:
    import torch
except ImportError:          # Part A does not need torch
    torch = None

# ---- Paths (relative to this notebook) ------------------------------------------------------------
RUN_DIR = Path("../emotion_vector_pipeline/output_drive")    # unzipped Drive archive (story_activations.npz, emotion_vectors.npz)
DATASET_ZIP = Path("../emotion_vector_pipeline/data/qwen3_8b_emotion_stories_1120_v1.zip")   # forbidden-term lists
PROMPTS_PATH = Path("prompts.json")
OUT_A = Path("outputs/A")

# ---- Shared settings (fixed by the method) ------------------------------------------------------
MAIN_LAYER = 23                      # notebook 03's reference block (0-based; about 2/3 of the way through 36 blocks)
SEED = 180                           # team seed from notebook 03, fixed before any result was seen
FOCUS = ["desperate", "calm"]        # the two emotions the later steering work centres on

# ---- Part A ---------------------------------------------------------------------------------------
N_PERM = 500
EXPECTED_SPLIT_SIZES = {"train": 784, "validation": 168, "test": 168}

# ---- Parts B/C ------------------------------------------------------------------------------------
MODEL_ID = "Qwen/Qwen3-8B"
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"   # same commit as the extraction run
C_EMOTIONS = ["desperate", "calm", "afraid", "sad", "happy"]
# "auto": run B/C with Qwen3-8B when a CUDA GPU is visible, otherwise skip them.
# "dryrun": Qwen3-0.6B on CPU with random vectors, a CODE CHECK ONLY. "skip": Part A only.
BC_MODE = os.environ.get("BC_MODE", "auto")
cuda_ok = torch is not None and torch.cuda.is_available()
if BC_MODE == "auto":
    BC_MODE = "full" if cuda_ok else "skip"
assert BC_MODE in {"full", "dryrun", "skip"}, BC_MODE
if BC_MODE == "full" and not cuda_ok:
    raise RuntimeError("BC_MODE='full' needs an NVIDIA CUDA GPU (use Colab).")
if BC_MODE == "dryrun" and torch is None:
    raise RuntimeError("BC_MODE='dryrun' needs torch and transformers installed.")
RUN_BC, DRY_RUN = BC_MODE != "skip", BC_MODE == "dryrun"
OUT_BC = Path("outputs/BC_DRYRUN_code_check_only" if DRY_RUN else "outputs/BC")
BC_LABEL = ("CODE CHECK ONLY: not results (Qwen3-0.6B, CPU, random vectors)" if DRY_RUN
            else f"Qwen3-8B NF4, block {MAIN_LAYER}")

OUT_A.mkdir(parents=True, exist_ok=True)
if RUN_BC:
    OUT_BC.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
print("RUN_DIR:", RUN_DIR.resolve())
print(f"BC_MODE: {BC_MODE} | CUDA available: {cuda_ok} | B/C label: {BC_LABEL if RUN_BC else 'skipped'}")
print("numpy", np.__version__, "| sklearn", sklearn.__version__, "| matplotlib", matplotlib.__version__,
      "| torch", getattr(torch, "__version__", "not installed"))

### 0.3 Plot style and small helpers
**What:**
- a consistent plot style;
- two save helpers, `save_fig` and `write_csv`;
- a tiny cell magic, `%%run_if`.

**Why:** every figure and table lands in `outputs/` with the same look, so they can go straight into the report. Cells that start with `%%run_if RUN_BC` run only when Parts B/C are enabled, and otherwise print "skipped". That lets *Run all* finish on a laptop without a GPU.

In [ ]:
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SERIES = {"desperate": "#2a78d6", "calm": "#eb6834", "afraid": "#1baf7a", "sad": "#eda100", "happy": "#e87ba4"}
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#184f95", "#6da7ec", "#f0efec", "#ec8a89", "#b3302f"])
SEQUENTIAL = LinearSegmentedColormap.from_list("blue_seq", ["#f6f9fd", "#9ec5f4", "#2a78d6", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False, "figure.dpi": 110,
})

def save_fig(fig, path):
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)

def write_csv(path, header, rows_):
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows_)
    print("saved", path)

from IPython import get_ipython
from IPython.core.magic import register_cell_magic

@register_cell_magic
def run_if(line, cell):
    # Run the cell only if the expression after %%run_if is true; otherwise say it was skipped.
    ip = get_ipython()
    if eval(line, ip.user_ns):
        ip.run_cell(cell).raise_error()
    else:
        print(f"skipped: {line.strip()} is False (BC_MODE = {BC_MODE!r}; Parts B/C need a CUDA GPU)")

---
# Part A: held-out stories

**Question:** the vectors were built from 784 stories about topics 0–13. Do they recognize the intended emotion in stories about **topics 14–19, which they never saw**?

**How:**
- **Score.** Each story's saved internal state at each block is scored against each of the 14 emotion vectors, giving 14 scores per story:
  `score = (story state − average state) · emotion vector`
  This measures how far the story points along that emotion's direction.
- **Two grades.** *Top-1:* is the highest score the right emotion? (Chance = 1/14.) *AUROC:* does the emotion's vector score its own stories above everyone else's? (Chance = 0.5.)
- **Two splits.** *Validation* (topics 14–16) is used for every block and every choice. *Test* (topics 17–19) is used only at the end, at blocks chosen beforehand.
- **Control.** Shuffle the training labels 500 times and rebuild "fake" vectors. The real vectors must beat them.

Order: check the data → check the code → check the in-sample result → validation → null → test. No later step changes an earlier choice.

### A.1 Load the archive and read its structure
**What:** loads the pooled story activations, shape `(1120 stories, 36 blocks, 4096 numbers)`, and the vectors, shape `(14 emotions, 36 blocks, 4096)`. It prints the keys, the shapes, and the emotion and block order **as stored in the files**.
**Why:** everything below indexes these arrays by position. If the emotion order were assumed rather than read (it is alphabetical in the file, not the order the dataset was generated in), every label would be silently wrong.

In [ ]:
ACT_PATH = RUN_DIR / "story_activations.npz"
VEC_PATH = RUN_DIR / "emotion_vectors.npz"
for p in (ACT_PATH, VEC_PATH):
    if not p.is_file():
        raise FileNotFoundError(f"Missing {p.resolve()}. Download the Drive archive "
                                "(emotion_vector_pipeline/README.md) and point RUN_DIR at it.")

with np.load(ACT_PATH, allow_pickle=False) as z:
    print("story_activations.npz keys:", z.files)
    pooled = z["pooled"]
    story_ids = z["story_ids"].astype(str)
    story_emotions = z["emotions"].astype(str)
    splits = z["splits"].astype(str)
    act_layers = z["layer_indices"].tolist()
with np.load(VEC_PATH, allow_pickle=False) as z:
    print("emotion_vectors.npz keys:  ", z.files)
    emotion_names = z["emotion_names"].astype(str).tolist()
    vec_layers = z["layer_indices"].tolist()
    unit_vectors = z["unit_vectors"]
    raw_vectors = z["raw_vectors"]
    center = z["center"]
    emotion_means = z["emotion_means"]
    saved_main = int(z["main_layer_index"]) if "main_layer_index" in z.files else None

N, L, D = pooled.shape
E = len(emotion_names)
print("pooled", pooled.shape, pooled.dtype, "| unit_vectors", unit_vectors.shape, "| center", center.shape)
print("emotion order (from file):", emotion_names)
print("layer order (from file):", act_layers)

### A.2 Verify shapes, labels and splits
**What:** asserts the expected shapes, that both files agree on block order, and that every vector has length 1. It then checks that the splits contain 784 / 168 / 168 stories with 56 / 12 / 12 per emotion, and, using `stories.jsonl`, that the splits really are by topic (0–13 / 14–16 / 17–19).
**Why:** the whole claim of Part A rests on "these stories are about topics the vectors never saw". This cell proves that is true for the data on disk, instead of trusting the README.

In [ ]:
assert pooled.shape == (1120, 36, 4096) and pooled.dtype == np.float32
assert unit_vectors.shape == raw_vectors.shape == emotion_means.shape == (E, L, D) == (14, 36, 4096)
assert center.shape == (L, D)
assert act_layers == vec_layers, "Layer order differs between the two files."
assert len(set(emotion_names)) == E
assert set(story_emotions) == set(emotion_names), "Story labels do not match emotion_names."
assert len(set(story_ids)) == N, "Duplicate story IDs."
assert np.isfinite(pooled).all() and np.isfinite(unit_vectors).all()
np.testing.assert_allclose(np.linalg.norm(unit_vectors, axis=-1), 1.0, atol=1e-5)
if saved_main is not None:
    assert saved_main == MAIN_LAYER, saved_main

LAYER_POS = {layer: i for i, layer in enumerate(act_layers)}   # block index -> array axis position
MAIN_POS = LAYER_POS[MAIN_LAYER]
FOCUS_IDX = [emotion_names.index(e) for e in FOCUS]
y = np.array([emotion_names.index(e) for e in story_emotions])  # integer labels in emotion_names order

split_idx = {s: np.flatnonzero(splits == s) for s in EXPECTED_SPLIT_SIZES}
for s, n in EXPECTED_SPLIT_SIZES.items():
    assert len(split_idx[s]) == n, (s, len(split_idx[s]))
    assert (np.bincount(y[split_idx[s]], minlength=E) == n // E).all(), s
print("Split sizes:", {s: len(i) for s, i in split_idx.items()},
      "| stories per emotion:", {s: n // E for s, n in EXPECTED_SPLIT_SIZES.items()})

stories_path = RUN_DIR / "stories.jsonl"
if stories_path.is_file():
    rows = [json.loads(line) for line in stories_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert [r["id"] for r in rows] == story_ids.tolist(), "stories.jsonl order differs from the activations"
    assert [r["emotion"] for r in rows] == story_emotions.tolist()
    assert [r["split"] for r in rows] == splits.tolist()
    topics = {s: sorted({r["topic_id"] for r in rows if r["split"] == s}) for s in EXPECTED_SPLIT_SIZES}
    assert topics == {"train": list(range(14)), "validation": [14, 15, 16], "test": [17, 18, 19]}, topics
    print("Topic split verified against stories.jsonl:", topics)
else:
    print("stories.jsonl not found; topic cross-check skipped.")

### A.3 Record provenance
**What:** stores the run ID, model commit, and SHA-256 fingerprints of both archive files.
**Why:** anyone can later confirm the reported numbers came from exactly these files. If a teammate re-extracts the vectors, the fingerprints change and the mismatch is visible.

In [ ]:
def sha256_of(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

manifest_path = RUN_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
provenance = {
    "run_id": manifest.get("run_id"),
    "model_revision": manifest.get("config", {}).get("model_revision"),
    "story_activations_sha256": sha256_of(ACT_PATH),
    "emotion_vectors_sha256": sha256_of(VEC_PATH),
    "emotion_names": emotion_names,
    "main_layer": MAIN_LAYER, "seed": SEED, "n_perm": N_PERM,
    "numpy": np.__version__, "sklearn": sklearn.__version__,
}
print(json.dumps(provenance, indent=1))

### A.4 Unit check: rebuild the vectors ourselves
**What:** recomputes the vectors from the 784 training stories with notebook 03's recipe:
1. average each emotion's stories;
2. subtract the average of those 14 averages;
3. scale each result to length 1.

It then compares the result with the saved vectors.
**Why:**
- **It proves we use exactly the vectors that notebook 03 made**, using training stories only. The match is exact (difference 0).
- **The null control in A.8 needs this function.** It must rebuild "fake" vectors the same way, and this cell proves the function is right.

In [ ]:
def build_vectors(acts, labels, n_emotions):
    # Notebook 03, "Emotion directions" cell: means -> subtract mean of means -> normalise.
    means = np.stack([acts[labels == e].mean(axis=0) for e in range(n_emotions)]).astype(np.float32)
    center_ = means.mean(axis=0)
    raw = means - center_[None]
    unit = raw / np.linalg.norm(raw, axis=-1, keepdims=True)
    return means, center_, raw, unit

train_idx = split_idx["train"]
X_train, y_train = pooled[train_idx], y[train_idx]
re_means, re_center, re_raw, re_unit = build_vectors(X_train, y_train, E)

rebuild_check = {}
for name, mine, saved in [("emotion_means", re_means, emotion_means), ("center", re_center, center),
                          ("raw_vectors", re_raw, raw_vectors), ("unit_vectors", re_unit, unit_vectors)]:
    np.testing.assert_allclose(mine, saved, rtol=1e-5, atol=1e-5, err_msg=name)
    rebuild_check[name] = float(np.abs(mine - saved).max())
    print(f"{name:14s} allclose OK   max |rebuilt - saved| = {rebuild_check[name]:.3g}")

### A.5 The score, and a check that it points the right way
**What:** defines `score = (state − center) · unit_vector` for all blocks, then runs two checks.
- **Synthetic check.** A fake story placed exactly along emotion *e*'s direction must score highest on *e* at every block.
- **Spot check.** One real score is recomputed by hand.

Finally it computes the scores for all 1,120 stories, giving shape `(1120, 36, 14)`.
**Why:** a transposed matrix or a sign error would still produce numbers, just meaningless ones. These checks make that kind of silent bug impossible.

In [ ]:
def score(acts_L, center_L, unit_L):
    # acts_L (n, D), center_L (D,), unit_L (E, D) -> (n, E) projection scores
    return (acts_L - center_L) @ unit_L.T

def score_all_layers(acts, center_, unit):
    # acts (n, L, D) -> (n, L, E)
    return np.stack([score(acts[:, l], center_[l], unit[:, l]) for l in range(acts.shape[1])], axis=1)

# Orientation check on synthetic points.
syn = score_all_layers(center[None] + 10.0 * unit_vectors, center, unit_vectors)   # (E, L, E)
assert syn.shape == (E, L, E)
assert (syn.argmax(-1) == np.arange(E)[:, None]).all(), "argmax does not return the planted emotion"
np.testing.assert_allclose(syn[np.arange(E), :, np.arange(E)], 10.0, rtol=1e-4)

scores = score_all_layers(pooled, center, unit_vectors)   # (N, L, E)
assert scores.shape == (N, L, E) and np.isfinite(scores).all()
i, e = 500, emotion_names.index("calm")
explicit = float(np.dot(pooled[i, MAIN_POS].astype(np.float64) - center[MAIN_POS], unit_vectors[e, MAIN_POS]))
assert np.isclose(scores[i, MAIN_POS, e], explicit, rtol=1e-4, atol=1e-3), (scores[i, MAIN_POS, e], explicit)
print("Synthetic orientation check OK | scores shape", scores.shape, "| spot check OK")

### A.6 The two grades: top-1 accuracy and AUROC
**What:**
- **Top-1:** the fraction of stories whose highest of the 14 scores is the right emotion. Chance is 1/14 = 0.071.
- **One-vs-rest AUROC per emotion:** pick one story of that emotion and one of any other emotion. AUROC is how often the emotion's vector scores the first one higher. 0.5 is a coin flip, 1.0 is perfect.
- **A fast AUROC for the null loop:** a vectorized version that handles ties correctly, checked here against scikit-learn on data with many ties.

**Why use both grades?** They answer different questions.
- Top-1 asks *"which of the 14 is it?"*, so it compares vectors with each other.
- AUROC asks *"does this one vector rise for its own emotion?"*, so it looks at one vector at a time.

In Part A these two disagree in an informative way.

In [ ]:
CHANCE = 1 / E

def top1_accuracy(s, labels):          # s: (n, E)
    return float((s.argmax(-1) == labels).mean())

def ovr_auroc(s, labels):              # sklearn reference, s: (n, E) -> (E,)
    return np.array([roc_auc_score(labels == e, s[:, e]) for e in range(s.shape[-1])])

def average_ranks(s):
    # 1-based ranks along axis 0; tied values share their average rank (as in Mann-Whitney / sklearn).
    n = s.shape[0]
    order = s.argsort(axis=0, kind="stable")
    srt = np.take_along_axis(s, order, axis=0)
    pos = np.arange(n).reshape((n,) + (1,) * (s.ndim - 1)) + np.zeros_like(srt, dtype=np.int64)
    starts = np.ones(srt.shape, bool); starts[1:] = srt[1:] != srt[:-1]
    ends = np.ones(srt.shape, bool); ends[:-1] = srt[1:] != srt[:-1]
    first = np.maximum.accumulate(np.where(starts, pos, 0), axis=0)
    last = np.flip(np.minimum.accumulate(np.flip(np.where(ends, pos, n - 1), 0), axis=0), 0)
    ranks = np.empty(srt.shape, np.float64)
    np.put_along_axis(ranks, order, (first + last) / 2 + 1, axis=0)
    return ranks

def ovr_auroc_fast(s, labels):
    # Mann-Whitney AUROC along axis 0 for every (layer, emotion) column; s: (n, L, E) -> (L, E).
    ranks = average_ranks(s)
    onehot = np.eye(s.shape[-1])[labels]                       # (n, E)
    n_pos = onehot.sum(0)                                      # (E,)
    rank_sum = np.einsum("nle,ne->le", ranks, onehot)
    return (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * (len(labels) - n_pos))

# Unit test with many forced ties (scores rounded to 1 decimal): must equal sklearn exactly.
_rng = np.random.default_rng(0)
_lab = np.repeat(np.arange(E), 12)
_s = np.round(_rng.normal(size=(len(_lab), 3, E)), 1)
assert (np.diff(np.sort(_s, axis=0), axis=0) == 0).any(), "test should contain ties"
np.testing.assert_allclose(ovr_auroc_fast(_s, _lab), np.stack([ovr_auroc(_s[:, l], _lab) for l in range(3)]), atol=1e-12)
print("tie-aware fast AUROC matches sklearn on a synthetic case with ties")

### A.7 Sanity check on the training stories (in-sample)
**What:** top-1 on the 784 stories that *built* the vectors. It must be well above chance (more than 5× chance, a threshold set in advance).
**Why:** this is the easiest possible test. If it failed, something upstream would be broken, and every later number would be meaningless. It is **not** a generalization result. A low value here also tells us something: picking the single highest score is a hard test even on familiar stories.

In [ ]:
train_scores = scores[train_idx]
train_acc = np.array([top1_accuracy(train_scores[:, l], y_train) for l in range(L)])
print(f"chance = {CHANCE:.3f}")
print(f"train top-1 at block {MAIN_LAYER}: {train_acc[MAIN_POS]:.3f} | min over blocks {train_acc.min():.3f} | max {train_acc.max():.3f}")
assert train_acc[MAIN_POS] > 5 * CHANCE, "Training accuracy is not well above chance; stop and investigate."

### A.8 Validation: all 36 blocks, and the choice of the "best" block
**What:** top-1 and AUROC for the 168 validation stories at every block. It also picks the **best validation block** with a fixed rule: highest top-1, ties broken by mean AUROC, then by the lower block. It confirms the fast AUROC equals scikit-learn on all 504 real (block, emotion) columns.
**Why:**
- **Where emotion is most readable.** Comparing blocks shows how deep in the model emotion becomes clearest.
- **The test blocks are fixed here.** Choosing the comparison block now, on validation only, means the test split can't influence any choice.

In [ ]:
val_idx = split_idx["validation"]
val_scores, y_val = scores[val_idx], y[val_idx]
val_acc = np.array([top1_accuracy(val_scores[:, l], y_val) for l in range(L)])
val_auroc = np.stack([ovr_auroc(val_scores[:, l], y_val) for l in range(L)])     # (L, E), sklearn
val_macro = val_auroc.mean(axis=1)

# Check the fast AUROC used in the null loop against sklearn on the real data.
np.testing.assert_allclose(ovr_auroc_fast(val_scores, y_val), val_auroc, rtol=0, atol=1e-12)
print("fast AUROC == sklearn AUROC on all", L * E, "validation (block, emotion) columns")

BEST_POS = sorted(range(L), key=lambda l: (-val_acc[l], -val_macro[l], act_layers[l]))[0]
BEST_LAYER = act_layers[BEST_POS]
print(f"block {MAIN_LAYER}: val top-1 {val_acc[MAIN_POS]:.3f}, macro AUROC {val_macro[MAIN_POS]:.3f}")
print(f"best validation block: {BEST_LAYER} (top-1 {val_acc[BEST_POS]:.3f}, macro AUROC {val_macro[BEST_POS]:.3f})")
for name, k in zip(FOCUS, FOCUS_IDX):
    print(f"  {name:10s} AUROC  block {MAIN_LAYER}: {val_auroc[MAIN_POS, k]:.3f} | block {BEST_LAYER}: {val_auroc[BEST_POS, k]:.3f}")

### A.9 Null control: are the real vectors better than "fake" ones?
**What:**
1. Shuffle the emotion labels among the training stories (still 56 per emotion).
2. Rebuild vectors with the same `build_vectors` checked in A.4.
3. Rescore the validation stories.
4. Repeat 500 times (seed 180).

A real result counts only if it beats the **95th percentile** of these fake results. The p-value is `(1 + number of fakes ≥ real) / 501`, so the smallest possible is 0.002.
**Why:** beating 1/14 isn't enough. Stories of the same emotion can share incidental features, and a random direction can separate a tight group of 12 stories by luck. This null measures how well *meaningless* vectors do on these exact stories, which is the bar the real ones must clear. It also first checks that the null code reproduces the real numbers when the labels are *not* shuffled.

In [ ]:
X_val = pooled[val_idx]

def null_metrics(train_labels):
    _, c, _, u = build_vectors(X_train, train_labels, E)
    s = score_all_layers(X_val, c, u)                                         # (168, L, E)
    acc = (s.argmax(-1) == y_val[:, None]).mean(0)                            # (L,)
    return acc, ovr_auroc_fast(s, y_val)                                      # (L,), (L, E)

acc_id, auc_id = null_metrics(y_train)
np.testing.assert_allclose(acc_id, val_acc, atol=1e-12)
np.testing.assert_allclose(auc_id, val_auroc, atol=1e-6)
print("null code path with unshuffled labels reproduces the real validation metrics")

rng = np.random.default_rng(SEED)
null_acc = np.empty((N_PERM, L))
null_auroc = np.empty((N_PERM, L, E))
t0 = time.time()
for p in range(N_PERM):
    null_acc[p], null_auroc[p] = null_metrics(rng.permutation(y_train))
    if (p + 1) % 100 == 0:
        print(f"  {p + 1}/{N_PERM} permutations, {time.time() - t0:.0f}s")

null_macro = null_auroc.mean(-1)                                  # (P, L)
acc_p95, macro_p95 = np.percentile(null_acc, 95, axis=0), np.percentile(null_macro, 95, axis=0)
auroc_p95 = np.percentile(null_auroc, 95, axis=0)                 # (L, E)
p_value = lambda null, real: (1 + (null >= real).sum(0)) / (1 + N_PERM)
acc_p, macro_p, auroc_p = p_value(null_acc, val_acc), p_value(null_macro, val_macro), p_value(null_auroc, val_auroc)

for pos in sorted({MAIN_POS, BEST_POS}):
    print(f"block {act_layers[pos]}: val top-1 {val_acc[pos]:.3f} vs null p95 {acc_p95[pos]:.3f} (null mean {null_acc[:, pos].mean():.3f}), p = {acc_p[pos]:.4f}")
    print(f"          macro AUROC {val_macro[pos]:.3f} vs null p95 {macro_p95[pos]:.3f}, p = {macro_p[pos]:.4f}")
print("blocks where val top-1 > null p95:", int((val_acc > acc_p95).sum()), "of", L)

### A.10 Save the validation tables
**What:** writes `validation_by_block.csv` (every block: train, validation and null numbers, and the AUROC per emotion). It also writes `validation_auroc_vs_null_block*.csv` (each emotion's AUROC against its own null).
**Why:** the report can quote exact numbers from files instead of from screenshots, and the README names these files as the source of every reported value.

In [ ]:
header = (["block", "train_top1", "val_top1", "null_top1_mean", "null_top1_p95", "top1_p",
           "val_macro_auroc", "null_macro_p95", "macro_p"]
          + [f"auroc_{e}" for e in emotion_names] + [f"null_p95_auroc_{e}" for e in emotion_names])
write_csv(OUT_A / "validation_by_block.csv", header, [
    [act_layers[l], round(train_acc[l], 4), round(val_acc[l], 4), round(null_acc[:, l].mean(), 4),
     round(acc_p95[l], 4), round(acc_p[l], 4), round(val_macro[l], 4), round(macro_p95[l], 4), round(macro_p[l], 4)]
    + [round(v, 4) for v in val_auroc[l]] + [round(v, 4) for v in auroc_p95[l]] for l in range(L)])

for pos in sorted({MAIN_POS, BEST_POS}):
    write_csv(OUT_A / f"validation_auroc_vs_null_block{act_layers[pos]}.csv",
              ["emotion", "val_auroc", "null_p95", "p", "above_null_p95"],
              [[e, round(val_auroc[pos, k], 4), round(auroc_p95[pos, k], 4), round(auroc_p[pos, k], 4),
                bool(val_auroc[pos, k] > auroc_p95[pos, k])] for k, e in enumerate(emotion_names)])

### A.11 Figure: accuracy by block
**What:** validation top-1 at every block (blue), the in-sample training curve (dashed) and the null's 5th–95th percentile band (gray). It marks block 23 and the best block.
**Why:** one picture shows the three key facts:
- the real vectors sit far above the null at every depth;
- emotion is most readable in the middle-to-late blocks;
- block 23 is close to the best.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.fill_between(act_layers, np.percentile(null_acc, 5, axis=0), acc_p95, color=GRID, alpha=0.9, lw=0,
                label="label-shuffle null (5th–95th pct)")
ax.plot(act_layers, train_acc, color=MUTED, lw=1.5, ls="--", label="train (in-sample)")
ax.plot(act_layers, val_acc, color=BLUE, label="validation (held-out topics)")
ax.axhline(CHANCE, color=MUTED, lw=1, ls=":")
ax.text(act_layers[-1], CHANCE + 0.01, "chance 1/14", color=INK2, ha="right", va="bottom", fontsize=9)
ax.axvline(MAIN_LAYER, color=INK2, lw=1)
ax.text(MAIN_LAYER + 0.3, 0.98, f"block {MAIN_LAYER}", color=INK2, va="top", fontsize=9)
ax.plot([BEST_LAYER], [val_acc[BEST_POS]], "o", ms=8, color=BLUE, mec=SURFACE, mew=2)
ax.annotate(f"best val: block {BEST_LAYER}\n{val_acc[BEST_POS]:.2f}", (BEST_LAYER, val_acc[BEST_POS]),
            xytext=(0, -34), textcoords="offset points", ha="center", fontsize=9, color=INK2)
ax.set(xlabel="decoder block (0-based)", ylabel="14-way top-1 accuracy", ylim=(0, 1.02), xlim=(-0.5, L - 0.5),
       title="Emotion-vector classification of stories by block (Qwen3-8B, validation split)")
ax.legend(loc="upper left", fontsize=9)
save_fig(fig, OUT_A / "val_accuracy_by_block.png"); plt.show()

### A.12 Figure: AUROC by block for desperate and calm
**What:** the AUROC curves for the two focus emotions, each with its own null 95th percentile (dashed), plus the 14-emotion average.
**Why:** the steering work will use these two vectors, so we check each one individually against its own null. The average can hide a weak vector.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for name, k, col in [(FOCUS[0], FOCUS_IDX[0], BLUE), (FOCUS[1], FOCUS_IDX[1], ORANGE)]:
    ax.plot(act_layers, val_auroc[:, k], color=col, label=f"{name}")
    ax.plot(act_layers, auroc_p95[:, k], color=col, lw=1, ls="--", alpha=0.8, label=f"{name}: null p95")
ax.plot(act_layers, val_macro, color=MUTED, lw=1.5, label="mean of 14 emotions")
ax.axhline(0.5, color=MUTED, lw=1, ls=":")
ax.axvline(MAIN_LAYER, color=INK2, lw=1)
ax.text(MAIN_LAYER + 0.3, 0.42, f"block {MAIN_LAYER}", color=INK2, fontsize=9)
ax.set(xlabel="decoder block (0-based)", ylabel="one-vs-rest AUROC", ylim=(0.4, 1.01), xlim=(-0.5, L - 0.5),
       title="Validation AUROC by block: desperate and calm")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), fontsize=9, ncol=3)
save_fig(fig, OUT_A / "val_auroc_by_block_desperate_calm.png"); plt.show()

### A.13 Figures: every emotion against its own null, and the null distribution
**What:**
- a bar per emotion (validation AUROC) with a black tick at that emotion's null 95th percentile. Bars that don't reach their tick did not beat the null.
- a histogram of the 500 fake accuracies at block 23, with the real accuracy marked.

**Why:**
- **The bars show which vectors are trustworthy individually**, since some emotions can pass while others fail.
- **The histogram makes the null result visible:** the real value sits far outside the fake ones.

In [ ]:
for pos in sorted({MAIN_POS, BEST_POS}):
    order = np.argsort(val_auroc[pos])
    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    colors = [BLUE if emotion_names[k] in FOCUS else "#b7b6ae" for k in order]
    ax.barh(range(E), val_auroc[pos, order] - 0.5, left=0.5, color=colors, height=0.7)
    ax.scatter(auroc_p95[pos, order], range(E), marker="|", s=160, color=INK, zorder=3, label="null p95")
    for r, k in enumerate(order):
        ax.text(val_auroc[pos, k] + 0.005, r, f"{val_auroc[pos, k]:.2f}", va="center", fontsize=8, color=INK2)
    ax.set_yticks(range(E), [emotion_names[k] for k in order])
    ax.axvline(0.5, color=MUTED, lw=1, ls=":")
    ax.set(xlim=(0.45, 1.04), xlabel="one-vs-rest AUROC (validation)",
           title=f"Per-emotion validation AUROC, block {act_layers[pos]}")
    ax.grid(axis="y", visible=False); ax.legend(loc="lower right", fontsize=9)
    save_fig(fig, OUT_A / f"val_auroc_per_emotion_block{act_layers[pos]}.png"); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.2))
n_val = len(val_idx)   # accuracy moves in steps of 1/168, so bins sit on those steps
ax.hist(null_acc[:, MAIN_POS], bins=(np.arange(0, round(null_acc[:, MAIN_POS].max() * n_val) + 2) - 0.5) / n_val,
        color="#b7b6ae", ec=SURFACE, lw=0.5)
ax.axvline(acc_p95[MAIN_POS], color=INK2, ls="--", lw=1)
ax.text(acc_p95[MAIN_POS], ax.get_ylim()[1] * 0.95, " null p95", color=INK2, fontsize=9, va="top")
ax.axvline(val_acc[MAIN_POS], color=BLUE)
ax.text(val_acc[MAIN_POS], ax.get_ylim()[1] * 0.95, f"real {val_acc[MAIN_POS]:.2f} ", color=BLUE, ha="right", fontsize=9, va="top")
ax.set(xlabel="validation top-1 accuracy", ylabel="permutations",
       title=f"Label-shuffle null, block {MAIN_LAYER} ({N_PERM} permutations)")
ax.grid(axis="x", visible=False)
save_fig(fig, OUT_A / f"null_accuracy_hist_block{MAIN_LAYER}.png"); plt.show()

### A.14 Test split: the final check, at blocks chosen beforehand
**What:** scores the 168 test stories (topics 17–19) at block 23 and at the best validation block. Both were fixed in A.8. It computes:
- top-1 accuracy;
- AUROC per emotion;
- a confusion matrix (true emotion vs. predicted emotion);
- a heatmap: true emotion × vector, mean test score, z-scored per vector with the *validation* mean and SD.

`outputs/A/test_lock.json` records which blocks were scored the first time. Rerunning recomputes the same numbers, but scoring different blocks stops with an error.
**Why:**
- **The honest number.** Validation was used to make a choice (the best block), so its numbers are slightly optimistic. The untouched test split gives the honest estimate.
- **Why z-score.** The heatmap puts every vector on a common scale, so you can see which emotions each vector responds to, regardless of its raw score level.

In [ ]:
TEST_LAYERS = [MAIN_LAYER] + ([BEST_LAYER] if BEST_LAYER != MAIN_LAYER else [])
lock_path = OUT_A / "test_lock.json"
if lock_path.is_file():
    locked = json.loads(lock_path.read_text())
    assert locked["test_layers"] == TEST_LAYERS, f"Test blocks changed after the test split was first scored: {locked}"
    print("test_lock.json found; same blocks as the first evaluation:", locked)
else:
    lock_path.write_text(json.dumps({"test_layers": TEST_LAYERS,
                                     "first_evaluated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
    print("wrote", lock_path)

test_idx = split_idx["test"]
test_scores, y_test = scores[test_idx], y[test_idx]
test_results = {}
for layer in TEST_LAYERS:
    pos = LAYER_POS[layer]
    s = test_scores[:, pos]
    pred = s.argmax(-1)
    auroc = ovr_auroc(s, y_test)
    cm = confusion_matrix(y_test, pred, labels=range(E))
    mu, sd = val_scores[:, pos].mean(0), val_scores[:, pos].std(0)
    z = (s - mu) / sd
    heat = np.stack([z[y_test == t].mean(0) for t in range(E)])           # (true, vector)
    test_results[layer] = dict(top1=top1_accuracy(s, y_test), auroc=auroc, cm=cm, heat=heat)
    print(f"TEST block {layer}: top-1 {test_results[layer]['top1']:.3f} (chance {CHANCE:.3f}), "
          f"macro AUROC {auroc.mean():.3f}, " + ", ".join(f"{e} {auroc[k]:.3f}" for e, k in zip(FOCUS, FOCUS_IDX)))
    write_csv(OUT_A / f"test_confusion_block{layer}.csv", ["true \\ predicted"] + emotion_names,
              [[emotion_names[t]] + cm[t].tolist() for t in range(E)])
    write_csv(OUT_A / f"test_heatmap_zscore_block{layer}.csv", ["true \\ vector"] + emotion_names,
              [[emotion_names[t]] + [round(v, 4) for v in heat[t]] for t in range(E)])

### A.15 Figure: test confusion matrix and z-scored heatmap
**What:**
- **Left:** how the 12 test stories of each emotion were classified. The diagonal is correct.
- **Right:** the z-scored heatmap, with the diagonal outlined.

**Why:**
- **The left panel shows *how* the vectors fail:** which emotions get confused with which, and whether a few vectors "win" too often.
- **The right panel shows whether each vector still prefers its own emotion** once score levels are evened out.

In [ ]:
def outline_diagonal(ax, n):
    for k in range(n):
        ax.add_patch(Rectangle((k - 0.5, k - 0.5), 1, 1, fill=False, ec=INK, lw=1.2))

for layer, res in test_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
    ax = axes[0]
    im = ax.imshow(res["cm"], cmap=SEQUENTIAL, vmin=0, vmax=12)
    for t in range(E):
        for p_ in range(E):
            if res["cm"][t, p_]:
                ax.text(p_, t, res["cm"][t, p_], ha="center", va="center", fontsize=8,
                        color="#ffffff" if res["cm"][t, p_] >= 7 else INK)
    ax.set_title(f"Confusion (counts of 12), top-1 {res['top1']:.2f}")
    ax.set_xlabel("predicted (argmax score)"); ax.set_ylabel("true emotion")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax = axes[1]
    vmax = float(np.abs(res["heat"]).max())
    im = ax.imshow(res["heat"], cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    outline_diagonal(ax, E)
    ax.set_title("Mean test score, z-scored per vector (validation mean/SD)")
    ax.set_xlabel("emotion vector"); ax.set_ylabel("true emotion")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="z")
    for ax in axes:
        ax.set_xticks(range(E), emotion_names, rotation=60, ha="right")
        ax.set_yticks(range(E), emotion_names)
        ax.grid(False)
    fig.suptitle(f"Test split (topics 17–19, n=168), block {layer}", fontsize=12)
    fig.tight_layout()
    save_fig(fig, OUT_A / f"test_confusion_heatmap_block{layer}.png"); plt.show()

### A.16 Save the Part A summary file
**What:** writes `outputs/A/summary.json` with the provenance, unit checks, validation numbers (with nulls) and test numbers.
**Why:** a single machine-readable record of Part A. The README and pull request quote their numbers from it.

In [ ]:
summary = {
    "provenance": provenance,
    "inputs": {"run_dir": str(RUN_DIR), "story_activations": str(ACT_PATH), "emotion_vectors": str(VEC_PATH)},
    "unit_checks": {"vector_rebuild_max_abs_diff": rebuild_check, "synthetic_orientation": "pass",
                    "fast_auroc_matches_sklearn": "pass", "null_identity_reproduces_real": "pass"},
    "chance_top1": CHANCE,
    "train_in_sample_top1": {"block_%d" % MAIN_LAYER: train_acc[MAIN_POS], "min": train_acc.min(), "max": train_acc.max()},
    "best_validation_block": BEST_LAYER,
    "validation": {f"block_{act_layers[pos]}": {
        "top1": val_acc[pos], "null_top1_mean": null_acc[:, pos].mean(), "null_top1_p95": acc_p95[pos], "top1_p": acc_p[pos],
        "macro_auroc": val_macro[pos], "null_macro_p95": macro_p95[pos], "macro_p": macro_p[pos],
        "auroc": dict(zip(emotion_names, val_auroc[pos])), "null_auroc_p95": dict(zip(emotion_names, auroc_p95[pos])),
        "auroc_p": dict(zip(emotion_names, auroc_p[pos]))} for pos in sorted({MAIN_POS, BEST_POS})},
    "validation_blocks_top1_above_null_p95": int((val_acc > acc_p95).sum()),
    "test": {f"block_{layer}": {"top1": r["top1"], "macro_auroc": r["auroc"].mean(),
                               "auroc": dict(zip(emotion_names, r["auroc"])),
                               "diagonal_is_row_max_in_heatmap": int((r["heat"].argmax(1) == np.arange(E)).sum())}
             for layer, r in test_results.items()},
}
def _py(o):
    if isinstance(o, dict): return {k: _py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_py(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 5)
    if isinstance(o, np.integer): return int(o)
    return o
(OUT_A / "summary.json").write_text(json.dumps(_py(summary), indent=2))
print("saved", OUT_A / "summary.json")

## Part A: final result
The next cell prints the result from *this* run. It also checks that the numbers quoted in "What it means" below still match, and warns if the data or code have changed since that text was written.

In [ ]:
res_main = test_results[MAIN_LAYER]
res_best = test_results.get(BEST_LAYER, res_main)
weak = [e for k, e in enumerate(emotion_names) if not val_auroc[MAIN_POS, k] > auroc_p95[MAIN_POS, k]]
pred_counts = res_main["cm"].sum(axis=0)
most_predicted = sorted(zip(emotion_names, pred_counts), key=lambda t: -t[1])[:3]

row = "{:<36s}{:>26s}{:>34s}"
print(f"PART A RESULT  (chance: top-1 = 1/14 = {CHANCE:.3f}; AUROC = 0.5)\n")
print(row.format("", f"block {MAIN_LAYER}", f"block {BEST_LAYER} (best on validation)"))
for label, f in [
    ("validation top-1 (null p95)",        lambda pos, r: f"{val_acc[pos]:.3f} ({acc_p95[pos]:.3f})"),
    ("validation macro AUROC (null p95)",  lambda pos, r: f"{val_macro[pos]:.3f} ({macro_p95[pos]:.3f})"),
    ("validation AUROC desperate / calm",  lambda pos, r: f"{val_auroc[pos, FOCUS_IDX[0]]:.3f} / {val_auroc[pos, FOCUS_IDX[1]]:.3f}"),
    ("TEST top-1",                         lambda pos, r: f"{r['top1']:.3f}"),
    ("TEST macro AUROC",                   lambda pos, r: f"{r['auroc'].mean():.3f}"),
    ("TEST AUROC desperate / calm",        lambda pos, r: f"{r['auroc'][FOCUS_IDX[0]]:.3f} / {r['auroc'][FOCUS_IDX[1]]:.3f}"),
]:
    print(row.format(label, f(MAIN_POS, res_main), f(BEST_POS, res_best)))
print(f"\nBlocks where validation top-1 beats the null p95: {int((val_acc > acc_p95).sum())} of {L}")
print(f"Emotions whose block-{MAIN_LAYER} validation AUROC does NOT beat their own null p95: {', '.join(weak) or 'none'}")
print(f"Training stories (in-sample) top-1 at block {MAIN_LAYER}: {train_acc[MAIN_POS]:.3f}")
print(f"Most-predicted emotions on test, block {MAIN_LAYER}: "
      + ", ".join(f"{e} {n}" for e, n in most_predicted) + f" (of {len(test_idx)} stories; an even spread would be 12 each)")

# The text below quotes the 2026-11-17 run. Warn if this run disagrees with it.
DOCUMENTED = {"val_top1_block23": 0.292, "val_top1_best": 0.321, "best_block": 21, "train_top1_block23": 0.463,
              "val_auroc_desperate_23": 0.842, "val_auroc_calm_23": 0.956, "test_top1_block23": 0.232,
              "test_macro_auroc_23": 0.802, "test_auroc_desperate_23": 0.786, "test_auroc_calm_23": 0.860}
live = {"val_top1_block23": val_acc[MAIN_POS], "val_top1_best": val_acc[BEST_POS], "best_block": BEST_LAYER,
        "train_top1_block23": train_acc[MAIN_POS], "val_auroc_desperate_23": val_auroc[MAIN_POS, FOCUS_IDX[0]],
        "val_auroc_calm_23": val_auroc[MAIN_POS, FOCUS_IDX[1]], "test_top1_block23": res_main["top1"],
        "test_macro_auroc_23": res_main["auroc"].mean(), "test_auroc_desperate_23": res_main["auroc"][FOCUS_IDX[0]],
        "test_auroc_calm_23": res_main["auroc"][FOCUS_IDX[1]]}
stale = {k: (DOCUMENTED[k], round(float(live[k]), 3)) for k in DOCUMENTED if abs(DOCUMENTED[k] - float(live[k])) > 0.0015}
print("\nThe interpretation below matches this run." if not stale
      else f"\nWARNING: the interpretation below quotes numbers that differ from this run (documented, live): {stale}")

### Part A: what it means
*(Numbers from the 2026-11-17 run, files in `outputs/A/`. The cell above re-checks them.)*

**1. The vectors generalize to new topics.** On stories about topics they never saw:
- validation top-1 is **0.292 at block 23** (4× chance), while 500 shuffled-label "fake" vectors reach only 0.119 at their 95th percentile;
- no fake reached the real value (p = 0.002, the smallest possible), and the same holds at **all 36 blocks**.

So the vectors carry information about emotion that topic or chance can't explain.

**2. The test split confirms it.** At block 23, test top-1 is **0.232** and mean AUROC is **0.802**.
- **Top-1 dropped from validation, as expected.** The best block was chosen on validation, and with 168 stories each story is worth 0.6 percentage points, so 0.29 → 0.23 is about 10 stories.
- **AUROC held up completely** (0.80 → 0.80). It is the more stable measure.

**3. The vectors detect well but don't compete well.** AUROC (~0.80) is much better than top-1 (~0.23–0.29).
- **Each vector rises for its own emotion,** but the vectors sit at different baseline levels, so a few win the "highest score" contest too often. On test, *inspired* and *surprised* were predicted 32 times each, against an even share of 12.
- **Desperate shows this most clearly:** test AUROC is 0.786, yet only 1 of its 12 test stories was *predicted* desperate.
- **Even in-sample top-1 is only 0.463.** So the weak top-1 comes from comparing raw scores across vectors, not mainly from the change of topic.

**4. Mistakes are "sensible".** Confusions stay inside emotion families: afraid → nervous, happy → inspired, hopeful → inspired/loving. The z-scored heatmap has a warm diagonal and two blocks, negative emotions versus positive ones. The vectors capture the broad emotional landscape more sharply than the fine distinctions within it.

**5. Block 23 is a good choice.** Accuracy rises through the middle of the model, peaks at block 21 (0.321), and falls in the last layers. Block 23 is within about 3 points of the peak and ties it on test.

**6. Desperate and calm are both valid detectors.**
- **Calm** has the **highest validation AUROC of all 14 (0.956)**; it is 0.860 on test.
- **Desperate** is 0.842 on validation and 0.786 on test.
- Both clear their own null comfortably.
- **Frustrated and hopeful do not beat their null** at block 23, so treat those two vectors with caution.

**Caveats:**
- **The same model (Qwen3-8B) wrote and read the stories.** The vectors may partly reflect Qwen's writing style for each emotion, which the topic split can't rule out; Part B is a check on this.
- **The labels are the emotions the stories were *meant* to show.** No human has verified them yet.
- **There are only 12 stories per emotion** per split, so small differences between emotions are noise.
- **Notebook 03 skipped the paper's neutral-text cleanup step.**

### A.17 Free memory before loading a model
**What:** deletes Part A's large arrays (about 1.5 GB). The small results stay for the final summary.
**Why:** on Colab, Parts B/C load an 8-billion-parameter model next, and freeing memory first avoids running out.

In [ ]:
for _name in ["pooled", "X_train", "X_val", "scores", "train_scores", "re_means", "re_center", "re_raw", "re_unit",
              "unit_vectors", "raw_vectors", "emotion_means", "null_auroc", "syn", "rows"]:
    globals().pop(_name, None)
gc.collect()
print("Part A large arrays released.")

---
# Parts B and C: shared setup (prompts, model, readout)

Parts B and C give the model **new text** and read its internal state, instead of reusing saved story states. The model only reads; it generates nothing and nothing is steered.

**Readout:**
1. Wrap the prompt as a chat user message.
2. Run Qwen3-8B on it.
3. Take block 23's output at the **last prompt token**, the point just before the model would start its answer.

**Score:** `cos(h − neutral baseline, emotion vector)`, where the neutral baseline is the average readout over 20 plain factual prompts. Cosine compares direction only, from −1 to +1.

Every cell from here on starts with `%%run_if RUN_BC`. Without a CUDA GPU, the cells print "skipped".

### BC.1 Load the prompts and check them for emotion words
**What:**
- loads `prompts.json`;
- builds a list of 101 forbidden words: the 14 emotion names and synonyms from the dataset manifest, plus a few extras;
- checks every B scenario, neutral prompt and filled-in C prompt, matching whole words and common endings ("calmly", "loved", "sadness");
- stops if any word is found;
- also checks the checker itself.

**Why:** Part B is only meaningful if the emotion is *implied*. If a prompt said "I'm desperate", the vector could just be detecting the word. This makes "no emotion words" a guarantee rather than a hope.

In [ ]:
%%run_if RUN_BC
prompts = json.loads(PROMPTS_PATH.read_text(encoding="utf-8"))
print("prompts.json status:", prompts.get("status"))
if str(prompts.get("status", "")).upper().startswith("DRAFT"):
    print("NOTE: prompts.json is still marked DRAFT (not yet reviewed). Results below are provisional.")
with zipfile.ZipFile(DATASET_ZIP) as zf:
    manifest_terms = json.loads(zf.read("manifest.json"))["config"]["terms"]
extra_terms = {k: v for k, v in prompts.get("extra_forbidden_terms", {}).items() if not k.startswith("_")}
forbidden = sorted({t.lower() for terms in [*manifest_terms.values(), *extra_terms.values()] for t in terms})
term_re = re.compile(r"\b(" + "|".join(map(re.escape, forbidden)) + r")(?:s|es|d|ed|ly|ing|ness)?\b", re.IGNORECASE)

def forbidden_hits(text):
    return sorted({m.group(0).lower() for m in term_re.finditer(text)})

# The checker itself must catch inflections and must not flag look-alikes.
assert forbidden_hits("She was Calmly sipping; we Loved it; sadness.") == ["calmly", "loved", "sadness"]
assert forbidden_hits("I made an average cake with a joystick.") == []

B, NEUTRAL, C = prompts["B_implicit"], prompts["neutral"], prompts["C_numerical"]
emotions_14 = sorted(manifest_terms)
to_check = [(p["id"], p["prompt"]) for p in B + NEUTRAL]
to_check += [(f'{t["id"]}[X={x}]', t["template"].format(X=x)) for t in C for x in t["values"]]
violations = {pid: hits for pid, text in to_check if (hits := forbidden_hits(text))}

ids = [p["id"] for p in B + NEUTRAL] + [t["id"] for t in C]
assert len(ids) == len(set(ids)), "duplicate prompt ids"
assert all(p["emotion"] in emotions_14 for p in B)
per_emotion = {e: sum(p["emotion"] == e for p in B) for e in emotions_14}
assert all(2 <= n <= 3 for n in per_emotion.values()), per_emotion
assert all("{X}" in t["template"] and 8 <= len(t["values"]) <= 10 for t in C)
print(f"{len(forbidden)} forbidden terms | checked {len(to_check)} texts "
      f"({len(B)} B, {len(NEUTRAL)} neutral, {len(to_check) - len(B) - len(NEUTRAL)} C)")
print("B scenarios per emotion:", per_emotion)
if violations:
    raise ValueError(f"Forbidden emotion terms found: {violations}")
print("No forbidden terms found.")

### BC.2 Load the block-23 vectors
**What:** loads the 14 emotion vectors at block 23, reading the emotion order from the file. In code-check mode, random vectors of the small model's size replace them later (BC.3).
**Why:** B and C use only the main block. Reading the order from the file again keeps the labels correct.

In [ ]:
%%run_if RUN_BC
VEC_PATH = RUN_DIR / "emotion_vectors.npz"
if VEC_PATH.is_file():
    with np.load(VEC_PATH, allow_pickle=False) as z:
        emotion_names = z["emotion_names"].astype(str).tolist()     # read the order, never assume it
        layer_indices = z["layer_indices"].tolist()
        unit_main = z["unit_vectors"][:, layer_indices.index(MAIN_LAYER)].astype(np.float32)   # (E, D)
    print("vectors:", VEC_PATH, unit_main.shape, "| order:", emotion_names)
elif DRY_RUN:
    emotion_names, unit_main = emotions_14, None
    print("emotion_vectors.npz not found; fine for the code check (random vectors are used anyway).")
else:
    raise FileNotFoundError(f"No vectors at {VEC_PATH.resolve()}. Set RUN_DIR to the unzipped Drive archive.")
assert set(emotion_names) == set(emotions_14)
E = len(emotion_names)

### BC.3 Load the model exactly as notebook 03 did
**What:** loads Qwen3-8B at the pinned commit `b968826…` in 4-bit NF4 (double quantization, fp16 compute), copied from notebook 03. Code-check mode loads Qwen3-0.6B on the CPU instead, with random unit vectors.
**Why:** the vectors only fit the internal states of the same model, at the same commit, with the same quantization. Any other setup would test a slightly different model.

In [ ]:
%%run_if RUN_BC
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

if DRY_RUN:
    MODEL_ID, DEVICE = "Qwen/Qwen3-0.6B", torch.device("cpu")
    from huggingface_hub import try_to_load_from_cache
    cached = try_to_load_from_cache(MODEL_ID, "config.json")        # .../snapshots/<commit>/config.json
    resolved_revision = Path(cached).parent.name if isinstance(cached, str) else "main"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=resolved_revision)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=resolved_revision).to(DEVICE, torch.float32)
else:
    DEVICE = torch.device("cuda:0")
    print("GPU:", torch.cuda.get_device_name(DEVICE))
    from huggingface_hub import HfApi
    from transformers import BitsAndBytesConfig
    resolved_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
    assert resolved_revision == MODEL_REVISION, resolved_revision
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=resolved_revision)
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=resolved_revision,
        quantization_config=quant_config,
        torch_dtype=torch.float16,
        device_map={'': str(DEVICE)},
        low_cpu_mem_usage=True,
        attn_implementation='sdpa',
    )
model.eval()
model.config.use_cache = False
blocks = model.model.layers
HIDDEN = model.config.hidden_size
assert 0 <= MAIN_LAYER < len(blocks)
print(f"{MODEL_ID} @ {resolved_revision} | blocks {len(blocks)} | hidden {HIDDEN} | "
      f"transformers {transformers.__version__} | torch {torch.__version__}")

if DRY_RUN:
    _rng = np.random.default_rng(SEED)
    unit_main = _rng.standard_normal((E, HIDDEN)).astype(np.float32)
    unit_main /= np.linalg.norm(unit_main, axis=1, keepdims=True)
    print("DRY RUN: random unit vectors", unit_main.shape, "(CODE CHECK ONLY)")
assert unit_main.shape == (E, HIDDEN), (unit_main.shape, HIDDEN)

### BC.4 Build the chat prompt and define the readout
**What:**
- **Formatting:** each prompt is wrapped with Qwen's chat template as a user message, with thinking turned off.
- **Readout:** `last_token_state` runs the model once and uses a temporary **hook** after block 23 to copy the state at the final prompt token.
- **Printout:** the cell prints one fully formatted prompt and its last token, which is `'\n\n'` after the empty `<think></think>` block.

**Why:**
- **Same reading position as the vectors.** The hook sits exactly where notebook 03 recorded the story states (the output of the block, before the next one).
- **A natural "how does the model see the situation" point.** The final token is where the model is about to answer.
- **A known difference from the paper**, which reads the `:` after "Assistant" in its own chat format. Both are "right before the answer".

In [ ]:
%%run_if RUN_BC
def build_prompt(p):
    return tokenizer.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True,
                                         enable_thinking=False, tokenize=False)

def encode(p):
    # The template already holds the special tokens, so add none.
    return tokenizer(build_prompt(p), add_special_tokens=False, return_tensors="pt")["input_ids"].to(DEVICE)

def last_token_state(p, layer=MAIN_LAYER):
    # Block `layer` output (notebook 03's hook point) at the final prompt token -> float32 (hidden,).
    ids = encode(p)
    captured = {}
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        assert hidden.ndim == 3 and hidden.shape[0] == 1
        captured["h"] = hidden[0, -1].float().cpu().numpy()
    handle = blocks[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            model.model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False, return_dict=True)
    finally:
        handle.remove()
    return captured["h"]

example = build_prompt(B[0]["prompt"])
ex_ids = encode(B[0]["prompt"])
print(repr(example))
print("tokens:", ex_ids.shape[1], "| last prompt token:", repr(tokenizer.decode(ex_ids[0, -1:])))
assert example.endswith("<|im_start|>assistant\n<think>\n\n</think>\n\n"), "unexpected chat-template ending"
assert tokenizer.decode(ex_ids[0]) == example, "tokenization does not round-trip"

### BC.5 Test the hook before trusting it
**What:** on one prompt, checks that:
1. the hook returns a vector of the model's width, `(hidden,)`;
2. the model's output is **identical** with and without the hook, and after it's removed;
3. the hook is really gone afterward;
4. the captured state equals the model's own `hidden_states` at block 23.

**Why:** a readout that changed the model, stayed attached, or read the wrong block would quietly corrupt every score in B and C. These checks confirm it only *reads*, and reads the right place.

In [ ]:
%%run_if RUN_BC
probe = NEUTRAL[0]["prompt"]
ids = encode(probe)
run = lambda: model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits.float().cpu()
with torch.inference_mode():
    logits_before = run()
    ref_state = model.model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False,
                            output_hidden_states=True).hidden_states[MAIN_LAYER + 1][0, -1].float().cpu().numpy()

# transformers 5.x leaves its own output-capture hook on each block after output_hidden_states=True,
# so "removed" means: our hook id is gone and the block's hook set is exactly what it was before.
hooks_before = set(blocks[MAIN_LAYER]._forward_hooks)
captured = {}
def _probe_hook(module, args, output):
    captured["h"] = (output[0] if isinstance(output, tuple) else output)[0, -1].float().cpu().numpy()
handle = blocks[MAIN_LAYER].register_forward_hook(_probe_hook)
our_hook_id = handle.id
assert our_hook_id in blocks[MAIN_LAYER]._forward_hooks
try:
    with torch.inference_mode():
        logits_during = run()
finally:
    handle.remove()
with torch.inference_mode():
    logits_after = run()

state = last_token_state(probe)
tol = dict(rtol=1e-5, atol=1e-5) if DRY_RUN else dict(rtol=1e-3, atol=1e-3)   # notebook 03 uses 1e-3 on GPU
assert state.shape == (HIDDEN,), state.shape
assert np.isfinite(state).all()
assert our_hook_id not in blocks[MAIN_LAYER]._forward_hooks, "hook was not removed"
assert set(blocks[MAIN_LAYER]._forward_hooks) == hooks_before, "hook set changed after last_token_state"
torch.testing.assert_close(logits_during, logits_before, **tol)
torch.testing.assert_close(logits_after, logits_before, **tol)
np.testing.assert_allclose(state, captured["h"], **tol)
np.testing.assert_allclose(state, ref_state, **tol)
hook_tests = {
    "state_shape": list(state.shape),
    "max_abs_logit_diff_with_hook": float((logits_during - logits_before).abs().max()),
    "max_abs_logit_diff_after_removal": float((logits_after - logits_before).abs().max()),
    "max_abs_diff_vs_hidden_states": float(np.abs(state - ref_state).max()),
    "our_hook_removed": our_hook_id not in blocks[MAIN_LAYER]._forward_hooks,
    "other_library_hooks_on_block": len(hooks_before),
}
print("hook tests passed:", hook_tests)

### BC.6 Neutral baseline and the cosine score
**What:** reads the 20 neutral prompts ("Convert 5 km to miles", …) and averages their states into a **neutral baseline**. Then it defines the score: `cos(state − baseline, emotion vector)`.
**Why:**
- **It strips out what every prompt shares.** All prompts share a large component that comes from the chat formatting itself, not the content. Subtracting the baseline leaves what is *specific* to each prompt.
- **Cosine ignores length.** It measures only direction, so prompts of different lengths are comparable.

The neutral prompts' own scores are kept as a reference for how big "nothing" looks.

In [ ]:
%%run_if RUN_BC
neutral_states = np.stack([last_token_state(p["prompt"]) for p in NEUTRAL])
neutral_baseline = neutral_states.mean(axis=0)
assert neutral_states.shape == (len(NEUTRAL), HIDDEN)

def cos_scores(h):
    # cos(h - neutral_baseline, unit_vectors[e, MAIN_LAYER]) for every emotion e -> (E,)
    d = h - neutral_baseline
    return (unit_main @ d) / (np.linalg.norm(unit_main, axis=1) * np.linalg.norm(d))

neutral_scores = np.stack([cos_scores(h) for h in neutral_states])   # reference spread only
print("neutral prompts' own scores: |cos| max", round(float(np.abs(neutral_scores).max()), 3))

---
# Part B: implicit scenarios

**Question:** given a short user message that *implies* an emotion without naming it, does that emotion's vector come out on top? For example: *"The release is in 40 minutes and every test in CI is still red…"* should score highest on **desperate**.

**Why this matters:** Part A used Qwen-written stories in a fixed style. Part B uses different text: short, first-person chat messages written by people (here, drafted for review). If the vectors still pick the right emotion, they track emotion itself, not the style of the training stories.

**Measurement:** for each of the 35 scenarios, rank the intended emotion among the 14 scores (1 = highest).
- **Top-1 hit rate:** chance is 1/14 = 0.07.
- **Top-3 hit rate:** chance is 3/14 = 0.21. It forgives scenarios that genuinely mix emotions.
- **Mean rank:** chance is 7.5.

### B.1 Score the 35 scenarios and rank the intended emotion
**What:** reads each scenario, computes its 14 cosine scores, and finds where the intended emotion ranks.
**Why:** the rank is the direct answer to "did the right emotion come out on top, or near the top?"

In [ ]:
%%run_if RUN_BC
B_states = np.stack([last_token_state(p["prompt"]) for p in B])
B_scores = np.stack([cos_scores(h) for h in B_states])                  # (n_B, E)
intended = np.array([emotion_names.index(p["emotion"]) for p in B])
own = B_scores[np.arange(len(B)), intended]
ranks = (B_scores > own[:, None]).sum(axis=1) + 1
assert B_scores.shape == (len(B), E) and np.isfinite(B_scores).all()

B_summary = {
    "label": BC_LABEL, "n_scenarios": len(B),
    "top1_hit_rate": float((ranks == 1).mean()), "top3_hit_rate": float((ranks <= 3).mean()),
    "mean_rank_of_intended": float(ranks.mean()),
    "chance": {"top1": 1 / E, "top3": 3 / E, "mean_rank": (E + 1) / 2},
    "per_emotion_mean_rank": {e: float(ranks[intended == k].mean()) for k, e in enumerate(emotion_names) if (intended == k).any()},
}

### B.2 Save the scores and draw the heatmap
**What:** writes `B_scores.csv` (every scenario's 14 scores and its rank, plus the neutral prompts for reference) and `B_summary.json`. It draws a heatmap of scenarios × emotion vectors, with the intended cell outlined.
**Why:** the heatmap is the same figure the paper uses (its Fig. 2). A strong outlined diagonal means the vectors find the implied emotion, and off-diagonal warm cells show which emotions get mixed up.

In [ ]:
%%run_if RUN_BC
order = np.lexsort((np.arange(len(B)), intended))                      # group rows by intended emotion
with open(OUT_BC / "B_scores.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "intended", "rank_of_intended"] + emotion_names)
    for i in order:
        w.writerow([B[i]["id"], B[i]["emotion"], int(ranks[i])] + [round(float(v), 5) for v in B_scores[i]])
    for i, p in enumerate(NEUTRAL):
        w.writerow([p["id"], "neutral", ""] + [round(float(v), 5) for v in neutral_scores[i]])
(OUT_BC / "B_summary.json").write_text(json.dumps(B_summary, indent=2))

fig, ax = plt.subplots(figsize=(10, 0.28 * len(B) + 2))
vmax = float(np.abs(B_scores).max())
im = ax.imshow(B_scores[order], cmap=DIVERGING, vmin=-vmax, vmax=vmax, aspect="auto")
for r, i in enumerate(order):
    ax.add_patch(Rectangle((intended[i] - 0.5, r - 0.5), 1, 1, fill=False, ec=INK, lw=1.2))
ax.set_xticks(range(E), emotion_names, rotation=60, ha="right")
ax.set_yticks(range(len(B)), [f'{B[i]["id"]} (rank {ranks[i]})' for i in order], fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="cos(h − neutral, vector)")
ax.set_title(f"Part B: implicit scenarios × emotion vectors (outlined = intended)\n{BC_LABEL} | "
             f"top-1 {B_summary['top1_hit_rate']:.2f}, top-3 {B_summary['top3_hit_rate']:.2f}, "
             f"mean rank {B_summary['mean_rank_of_intended']:.1f}", fontsize=10)
fig.tight_layout()
save_fig(fig, OUT_BC / "B_heatmap.png"); plt.show()

## Part B: final result

In [ ]:
%%run_if RUN_BC
tag = "[CODE CHECK ONLY: NOT RESULTS] " if DRY_RUN else ""
n = len(B)
k1, k3 = int((ranks == 1).sum()), int((ranks <= 3).sum())
status = str(prompts.get("status", "")).split(",")[0]
print(f"{tag}PART B RESULT  ({BC_LABEL}; prompts: {status})\n")
print(f"  Top-1 hit rate : {k1}/{n} = {k1/n:.2f}   (chance {1/E:.2f}; {k1/n*E:.1f}x chance)")
print(f"  Top-3 hit rate : {k3}/{n} = {k3/n:.2f}   (chance {3/E:.2f})")
print(f"  Mean rank of the intended emotion: {ranks.mean():.1f} of {E}   (1 = best; chance {(E+1)/2:.1f})")
for e in FOCUS:
    k = emotion_names.index(e)
    print(f"  {e:10s} scenarios ranked their own emotion: {ranks[intended == k].tolist()}")
winners = Counter(emotion_names[j] for j in B_scores.argmax(1))
print("  Emotions that most often score highest: " + ", ".join(f"{e} {c}" for e, c in winners.most_common(3))
      + f"   (an even spread would be about {n/E:.1f} each)")
if DRY_RUN:
    print("\n  These numbers come from random vectors on a small model. They only show that the code runs.")

### Part B: what it means
*(Status as of this commit: Part B has run only as a code check. The real result comes from running this notebook on a Colab GPU, and the cell above then prints it.)*

**How to read the result:**
- **Strong:** top-1 well above 0.07 *and* top-3 well above 0.21, mean rank near 1–3, and desperate and calm scenarios ranking their own emotion first or second. Then the vectors recognize emotion in a completely different kind of text, which is evidence they track emotion itself, not the style of the training stories.
- **Mixed:** top-1 low but top-3 high means the vectors land on the right *family* of emotions (as in Part A's confusions) but not always the exact one. Check the heatmap to see whether the misses are near neighbors (afraid ↔ nervous) or unrelated.
- **Bias:** if one emotion "wins" far more often than about 2.5 times in 35, that vector sits at a higher baseline (the same calibration effect seen in Part A). That is a property of the vectors, not evidence about the prompts.
- **Weak:** top-1 and top-3 near chance. Then the story-trained vectors don't transfer to chat messages, a sign they partly encode the stories' style.

**Caveats:**
- **The prompts are a draft.** The 35 scenarios were drafted by Claude Code and are awaiting review.
- **Some scenarios mix emotions.** For example, the biopsy wait could read as afraid as well as nervous. A "miss" can be prompt ambiguity rather than vector failure.
- **The intended labels are our judgment.** Having teammates label the scenarios blind would show how clear each one is.
- **Two choices that could shift scores:** the neutral baseline uses factual Q&A prompts, and the readout token differs from the paper's.
- **The paper's version is qualitative.** It used 12 scenarios and only described a "strong diagonal", so any comparison with it is qualitative.

---
# Part C: numerical prompts

**Question:** when a prompt stays the same except for one number, do the emotion scores move with how dire that number makes the situation? This is the paper's Figure 3: e.g. *"I just took {X} mg of Tylenol for my back pain"*, where "afraid" should rise and "calm" fall as the dose becomes dangerous.

**Why this matters:** the words barely change, only the number does. If the scores still move in the right direction, the model represents the *meaning* of the situation (8,000 mg is dangerous), not just the words.

**Measurement:** for each template and emotion, **Spearman ρ** between X and the score. It runs from −1 to +1: +1 means the score rises with every increase in X, and 0 means no consistent trend. It only uses the *order* of the values, so uneven X spacing doesn't matter.

**Control:** *"I just ate {X} grapes."* The number changes but nothing emotional should. Any trend there shows how much the scores move just because a number changed.

### C.1 Write down the expected directions *before* looking
**What:** for each template, the direction each emotion should move as X grows.
- **Paper's directions:** for Tylenol, the missing dog and startup runway, they come from the paper's text.
- **Our own templates:** for code failures and rent, they are Claude Code's proposal. Edit them before the real run if you disagree.
- **Grapes:** expects no trend.

**Why:** stating predictions before the run keeps the interpretation honest. With 30 values of ρ to look at, some will look interesting by chance.

In [ ]:
%%run_if RUN_BC
# +1 = score should rise as X grows, -1 = should fall. "paper": Sofroniew et al. 2026 §1.2 / Fig. 3 text.
# "proposed": Claude Code's proposal for our added templates (edit before the real run if you disagree).
C_EXPECTED = {
    "tylenol_mg":            {"source": "paper",    "afraid": +1, "calm": -1},
    "dog_missing_days":      {"source": "paper",    "sad": +1},
    "startup_runway_months": {"source": "paper",    "afraid": -1, "sad": -1, "calm": +1},
    "code_failed_tests":     {"source": "proposed", "desperate": +1, "calm": -1},
    "rent_due_days":         {"source": "proposed", "desperate": -1, "afraid": -1, "calm": +1},
    "grapes_control":        {"source": "control"},   # no emotion should trend
}
for t, exp in C_EXPECTED.items():
    print(f"{t:22s} [{exp['source']:8s}] " + (", ".join(f"{e} {'rises' if s > 0 else 'falls'}" for e, s in exp.items() if e != "source") or "no trend expected"))

### C.2 Score every template at every value, compute ρ, and plot
**What:** reads all 60 prompts (6 templates × 10 values) and computes the 14 scores for each. It then computes Spearman ρ for the five focus emotions, saves `C_scores.csv` and `C_spearman.csv`, and plots one panel per template.
**Why:** the plots show the shape of each trend (smooth, a jump, flat), and ρ summarizes each line as a single number for the result table.

In [ ]:
%%run_if RUN_BC
C_rows, C_rho = [], []
C_scores = {}
for t in C:
    S = np.stack([cos_scores(last_token_state(t["template"].format(X=x))) for x in t["values"]])   # (n_X, E)
    assert S.shape == (len(t["values"]), E)
    C_scores[t["id"]] = S
    for x, row in zip(t["values"], S):
        C_rows.append([t["id"], x, t["template"].format(X=x)] + [round(float(v), 5) for v in row])
    for e in C_EMOTIONS:
        rho, p = spearmanr(t["values"], S[:, emotion_names.index(e)])
        C_rho.append({"template": t["id"], "emotion": e, "spearman_rho": round(float(rho), 4),
                      "p": round(float(p), 4), "n": len(t["values"])})

with open(OUT_BC / "C_scores.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["template", "X", "prompt"] + emotion_names); w.writerows(C_rows)
with open(OUT_BC / "C_spearman.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(C_rho[0])); w.writeheader(); w.writerows(C_rho)

ncol = 3
nrow = int(np.ceil(len(C) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.8 * nrow), squeeze=False)
for ax, t in zip(axes.flat, C):
    S, xs = C_scores[t["id"]], np.array(t["values"], float)
    rhos = []
    for e in C_EMOTIONS:
        rho = next(r["spearman_rho"] for r in C_rho if r["template"] == t["id"] and r["emotion"] == e)
        ax.plot(xs, S[:, emotion_names.index(e)], "-o", ms=4, color=SERIES[e], label=e)
        rhos.append(f"{e} {rho:+.2f}")
    ax.set_xscale("log")
    ax.axhline(0, color=MUTED, lw=1, ls=":")
    ax.set_title(t["template"].replace("{X}", "X") + "\nSpearman ρ: " + " · ".join(rhos), fontsize=8.5)
    ax.set_xlabel("X (log scale)"); ax.set_ylabel("cos score")
for ax in list(axes.flat)[len(C):]:
    ax.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(C_EMOTIONS), fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Part C: numerical templates, {BC_LABEL}", fontsize=11)
fig.tight_layout(rect=(0, 0.03, 1, 1))
save_fig(fig, OUT_BC / "C_lines.png"); plt.show()

### C.3 Check each expected direction
**What:** for every expected (template, emotion) pair, three checks:
1. Is ρ's **sign** right?
2. Is |ρ| beyond what random orderings of 10 points reach 95% of the time? That is about 0.65, simulated here.
3. Is |ρ| bigger than the **same emotion's |ρ| on grapes**?

"Supported" requires all three.
**Why:** a high ρ alone isn't enough. Changing a number can shift the whole internal state steadily, so even meaningless directions can show strong trends. In the code check, random vectors often passed check 2, even on grapes. Check 3, against grapes, helps.
**But it isn't enough on its own.** In the code check (random vectors, small model), 3 of 11 expectations still came out "supported" purely by chance. So treat "supported" as *necessary but not sufficient*: look for a consistent pattern across templates, especially opposite directions.

In [ ]:
%%run_if RUN_BC
def spearman_cutoff(n, sims=20000, seed=SEED):
    # |rho| that a random ordering of n distinct points exceeds only 5% of the time (two-sided)
    perms = np.argsort(np.random.default_rng(seed).random((sims, n)), axis=1)
    d = perms - np.arange(n)
    rho = 1 - 6 * (d ** 2).sum(1) / (n * (n ** 2 - 1))
    return float(np.percentile(np.abs(rho), 95))

rho_of = {(r["template"], r["emotion"]): r["spearman_rho"] for r in C_rho}
C_checks = []
for t in C:
    cut = spearman_cutoff(len(t["values"]))
    for e, sign in C_EXPECTED.get(t["id"], {}).items():
        if e == "source":
            continue
        rho, grapes = rho_of[(t["id"], e)], rho_of.get(("grapes_control", e), 0.0)
        right_sign, strong, beats_control = np.sign(rho) == sign, abs(rho) >= cut, abs(rho) > abs(grapes)
        verdict = ("supported" if right_sign and strong and beats_control
                   else "right direction, not conclusive" if right_sign else "NOT supported")
        C_checks.append({"template": t["id"], "source": C_EXPECTED[t["id"]]["source"], "emotion": e,
                         "expected": "rises" if sign > 0 else "falls", "rho": rho, "grapes_rho": grapes,
                         "cutoff": round(cut, 3), "verdict": verdict})
write_csv(OUT_BC / "C_expected_direction_checks.csv", list(C_checks[0]), [list(c.values()) for c in C_checks])

## Part C: final result

In [ ]:
%%run_if RUN_BC
tag = "[CODE CHECK ONLY: NOT RESULTS] " if DRY_RUN else ""
print(f"{tag}PART C RESULT  ({BC_LABEL})\n")
print(f"  {'template':22s} {'source':9s} {'emotion':10s} {'expected':9s} {'rho':>6s} {'grapes':>7s}  verdict")
for c in C_checks:
    print(f"  {c['template']:22s} {c['source']:9s} {c['emotion']:10s} {c['expected']:9s} {c['rho']:+6.2f} {c['grapes_rho']:+7.2f}  {c['verdict']}")
n_sup = sum(c["verdict"] == "supported" for c in C_checks)
print(f"\n  Expected directions supported: {n_sup} of {len(C_checks)} "
      f"(paper's templates: {sum(c['verdict'] == 'supported' for c in C_checks if c['source'] == 'paper')} of "
      f"{sum(c['source'] == 'paper' for c in C_checks)})")
print("  Grapes control rho (should be near 0): " + ", ".join(f"{e} {rho_of[('grapes_control', e)]:+.2f}" for e in C_EMOTIONS))
if DRY_RUN:
    print("\n  These numbers come from random vectors on a small model. They only show that the code runs.")

### Part C: what it means
*(Status as of this commit: Part C has run only as a code check. The real result comes from running this notebook on a Colab GPU, and the cell above then prints it.)*

**How to read the result:**
- **Strong:** most expected directions are "supported", especially the paper's templates (afraid ↑ and calm ↓ with Tylenol dose; sad ↑ with days the dog is missing; afraid and sad ↓ and calm ↑ with months of runway), *and* the grapes ρ values are clearly smaller. Then, as the paper concludes, the vectors follow the *meaning* of the number, not the number itself.
- **Opposite directions are the most convincing evidence.** Afraid rising with Tylenol dose but *falling* with runway months is hard to explain as "the number got bigger".
- **Grapes trends as strongly as the real templates:** the trends come mostly from the number changing, not from emotion. Treat the C results as inconclusive.
- **"Right direction, not conclusive"** means the sign matched but the trend was weak or no bigger than on grapes. With only 10 points per line that is common, so it isn't a failure by itself.

**Caveats:**
- **Only the Tylenol wording matches the paper.** The dog and runway templates are paraphrases (their exact wording isn't in the paper's text), and the X values are ours.
- **Three of the paper's six templates aren't included:** food and drink, a sister's age, and exam results.
- **Many tests at once.** 30 ρ values are computed, so one or two can look striking by luck.
- **Random vectors can pass.** In the code check, random vectors earned "supported" on 3 of 11 expectations. Comparing against many random directions would be a stronger control. It is not implemented; that is your decision.
- **The paper reports no numbers here.** It only describes the trends, so any comparison with it is qualitative.

---
# Final summary and run record
**What:** one table of the headline results from all three parts, and a record of how B and C were run (model, commit, library versions, hook tests, prompt-file fingerprint and status).
**Why:** the summary is what goes into the report. The run record lets anyone confirm later which model and prompts produced the B and C numbers.

In [ ]:
%%run_if RUN_BC
run_info = {
    "notebook": "ABC_validation_walkthrough.ipynb", "label": BC_LABEL, "bc_mode": BC_MODE, "model_id": MODEL_ID,
    "model_revision": resolved_revision, "main_layer": MAIN_LAYER, "device": str(DEVICE),
    "transformers": transformers.__version__, "torch": torch.__version__,
    "prompts_sha256": hashlib.sha256(PROMPTS_PATH.read_bytes()).hexdigest(), "prompts_status": prompts.get("status"),
    "n_prompts": {"B": len(B), "neutral": len(NEUTRAL), "C": sum(len(t["values"]) for t in C)},
    "hook_tests": hook_tests, "emotion_names": emotion_names,
    "vectors": "random unit vectors (code check)" if DRY_RUN else str(VEC_PATH),
}
(OUT_BC / "run_info.json").write_text(json.dumps(run_info, indent=2))
print("saved", OUT_BC / "run_info.json")

In [ ]:
r23 = test_results[MAIN_LAYER]
print("OVERALL SUMMARY\n")
print(f"Part A (real; Qwen3-8B vectors, run {provenance['run_id']}):")
print(f"  validation top-1 {val_acc[MAIN_POS]:.3f} at block {MAIN_LAYER} vs shuffled-label null p95 {acc_p95[MAIN_POS]:.3f} (chance {CHANCE:.3f});"
      f" beats the null at {int((val_acc > acc_p95).sum())}/{L} blocks")
print(f"  test top-1 {r23['top1']:.3f}, test macro AUROC {r23['auroc'].mean():.3f};"
      f" desperate {r23['auroc'][FOCUS_IDX[0]]:.3f}, calm {r23['auroc'][FOCUS_IDX[1]]:.3f}")
if not RUN_BC:
    print("\nParts B and C: NOT RUN here (no CUDA GPU; BC_MODE='skip'). Run this notebook on a Colab GPU for real results.")
elif DRY_RUN:
    print("\nParts B and C: CODE CHECK ONLY (Qwen3-0.6B, random vectors). The code ran end to end; the numbers are NOT results.")
else:
    print(f"\nPart B (real; prompts: {str(prompts.get('status', '')).split(',')[0]}):")
    print(f"  top-1 {B_summary['top1_hit_rate']:.2f} (chance {1/E:.2f}), top-3 {B_summary['top3_hit_rate']:.2f} (chance {3/E:.2f}),"
          f" mean rank {B_summary['mean_rank_of_intended']:.1f} (chance {(E+1)/2:.1f})")
    print("Part C (real):")
    print(f"  expected directions supported: {sum(c['verdict'] == 'supported' for c in C_checks)} of {len(C_checks)}")

### Overall: what it means
- **Part A (done)** shows the emotion vectors recognize emotion in stories about unseen topics, far beyond a strict null. Calm and desperate are both reliable detectors. The weakness is calibration across vectors (top-1), not detection (AUROC).
- **Part B** tests whether that recognition carries over to a different kind of text (chat messages with the emotion implied).
- **Part C** tests whether the vectors follow the *meaning* of a situation as one number changes.

Together, a strong B and C would show the vectors track emotion broadly and not quirks of Qwen's stories. Weak B or C would point to the stories' style as part of what the vectors encode.

None of these tests shows the vectors *cause* behavior. That is the job of the steering experiments, which these results support by confirming the desperate and calm vectors are sound detectors at block 23.